In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
import matplotlib as mpl
import warnings; warnings.simplefilter('ignore')
import os
import sys
import h5py
import pandas as pd
import seaborn as sns
sys.path.insert(0, '/Users/jsmonzon/Research/SatGen/mcmc/src/')
import jsm_ancillary
import jsm_visualize
import jsm_SHMR
import jsm_mcmc
import jsm_stats
import jsm_simload
import evolve as ev
import galhalo as gh
import profiles as profiles
import config as cfg

In [3]:
plt.style.use('../../../SatGen/notebooks/paper1/paper.mplstyle')
double_textwidth = 7.0 #inches
single_textwidth = 3.5 #inches
levelz = [1-0.99, 1-0.95, 1-0.68]

In [ ]:
class Discrete_MassBin:
    """
    Wraps a single sample file and computes binned/averaged subhalo
    statistics (Nacc, SHMF, fsub) across trees, storing all results
    in a single `self.results` dictionary.
    """

    def __init__(self, filename, regime, order="all", n_bins=30, **kwargs):
        self.filename = filename
        self.regime = regime
        self.order = order
        self.n_bins = n_bins

        for key, value in kwargs.items():
            setattr(self, key, value)

        self._ii = None        # cached loaded sample
        self.results = {}      # all computed stats live here
        self.average_Nacc()
        self.average_SHMF()
    # ------------------------------------------------------------------
    # data loading / helpers
    # ------------------------------------------------------------------
    @property
    def order_key(self):
        return f"k{self.order}" if isinstance(self.order, int) else "all"

    def load_data(self, force=False):
        """Load (and cache) the sample dataframe."""
        if self._ii is None or force:
            self._ii = jsm_ancillary.load_sample(self.filename)
        return self._ii

    # ------------------------------------------------------------------
    # stats
    # ------------------------------------------------------------------
    def average_Nacc(self):
        ii = self.load_data()
        Nsub_mat = jsm_ancillary.make_matrix(ii, f"Nsub_{self.regime}_{self.order_key}")  # (n_trees, n_t)

        logMvir = np.log10(jsm_ancillary.make_matrix(ii, "MAH")[:, 0])
        logc = np.log10(jsm_ancillary.make_matrix(ii, "host_c")[:, 0])
        log1pz50 = np.log10(1 + ii.host_z50.values)

        self.results.update({
            "logMvir": logMvir[0],
            "log1pz50": log1pz50,
            "logc": logc,
            "Nsub_mat": Nsub_mat,
            "Nsub_ave": np.mean(Nsub_mat, axis=0),
            "Nsub_std": np.std(Nsub_mat, axis=0),
        })

    def average_SHMF(self):
        ii = self.load_data()
        submass = ii[f"submass_{self.regime}_{self.order_key}"].values  # (n_trees,) of arrays
        host_mass = jsm_ancillary.make_matrix(ii, "MAH")[:, 0]  # (n_trees,) at z=0

        n_total = len(submass)
        ratio_list = [
            m_arr / M for m_arr, M in zip(submass, host_mass)
            if len(m_arr) > 0 and M > 0
        ]
        n_lost = n_total - len(ratio_list)
        print(f"[average_SHMF] dropped {n_lost}/{n_total} trees (empty submass or zero host mass)")

        bin_edges = np.linspace(-3, 0, self.n_bins + 1)
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
        dlog = bin_edges[1] - bin_edges[0]

        counts = np.zeros((len(ratio_list), self.n_bins))
        for i, ratios_i in enumerate(ratio_list):
            log_r = np.log10(ratios_i[ratios_i > 0])
            counts[i], _ = np.histogram(log_r, bins=bin_edges)

        counts = counts / dlog  # dN/dlog(m/M)

        self.results.update({
            "log_mM": bin_centers,
            "shmf_ave": np.mean(counts, axis=0),
            "shmf_std": np.std(counts, axis=0),
        })

    # def average_facc(self):
    #     ii = self.load_data()
    #     fsub_mat = jsm_ancillary.make_matrix(ii, f"fsub_{self.regime}_{self.order_key}")  # (n_trees, n_t)

    #     self.results.update({
    #         "fsub_ave": np.mean(fsub_mat, axis=0),
    #         "fsub_p16": np.percentile(fsub_mat, 16, axis=0),
    #         "fsub_p84": np.percentile(fsub_mat, 84, axis=0),
    #     })

In [ ]:
files = []
for file in os.listdir("../../data/vdb/"):
    if not file.endswith(".h5"):
        continue
    files.append(file)

files = np.sort(files)

datasets = []
for file in files:
    datasets.append(Discrete_MassBin("../../data/vdb/"+file, regime="artificial"))

In [ ]:
datasets[1].results["logMvir"]

In [ ]:
fig, axes = plt.subplots(2,4, figsize=(2*double_textwidth, double_textwidth), sharex=True, sharey=True)

axes[0,0].plot(np.log10(1+cfg.zsample), testr["Nsub_ave"], color="C1", label='within R$_{\\rm vir}$')

axes[0,0].set_yscale("log")
axes[0,0].set_xlim(0,1)
axes[0,0].set_ylim(1e-2, 1e2)

axes[0,0].set_ylabel("< N$_{\\rm sub}$ >")
axes[1,0].set_ylabel("< N$_{\\rm sub}$ >")

axes[1,0].set_xlabel("log (1+z)")
axes[1,1].set_xlabel("log (1+z)")
axes[1,2].set_xlabel("log (1+z)")
axes[1,3].set_xlabel("log (1+z)")

plt.tight_layout()
plt.show()

In [ ]:
testw = average_Nacc("../../data/vdb/13.4_files.h5", regime="withering")
testr = average_Nacc("../../data/vdb/13.4_files.h5", regime="rvir")
testa = average_Nacc("../../data/vdb/13.4_files.h5", regime="artificial")

In [ ]:
plt.plot(np.log10(1+cfg.zsample), testw["Nsub_ave"], color="C0", label='survivors')
plt.plot(np.log10(1+cfg.zsample), testr["Nsub_ave"], color="C1", label='within R$_{\\rm vir}$')
plt.plot(np.log10(1+cfg.zsample), testa["Nsub_ave"], color="C2", label='resiliant to\nartificial disruption')

plt.fill_between(np.log10(1+cfg.zsample), testw["Nsub_ave"] - testw["Nsub_std"], testw["Nsub_ave"] + testw["Nsub_std"], alpha=0.1, color="C0")
plt.fill_between(np.log10(1+cfg.zsample), testr["Nsub_ave"] - testr["Nsub_std"], testr["Nsub_ave"] + testr["Nsub_std"], alpha=0.1, color="C1")
plt.fill_between(np.log10(1+cfg.zsample), testa["Nsub_ave"] - testa["Nsub_std"], testa["Nsub_ave"] + testa["Nsub_std"], alpha=0.1, color="C2")

plt.yscale("log")
plt.xlim(0,1)
plt.ylim(1e-2, 1e2)
plt.legend(framealpha=1)

plt.xlabel("log (1+z)")
plt.ylabel("< N$_{\\rm sub}$ >")
plt.show()

In [ ]:
def MAH_split(MAH_mat, sort_arr, arr_label, splitsize=200):

    sort_mask = sort_arr.argsort()
    arr_sorted = sort_arr[sort_mask]
    MAHs_sorted = MAH_mat[sort_mask]

    num_bins = MAH_mat.shape[0] // splitsize

    arr_meds = []
    for i in range(0, len(arr_sorted), splitsize):
        med = np.average(arr_sorted[i:i+splitsize])
        arr_meds.append(med)

    arr_meds = np.array(arr_meds)

    # --- colormap ---
    norm = mpl.colors.Normalize(vmin=arr_meds.min(), vmax=arr_meds.max())
    cmap = mpl.cm.coolwarm

    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])


    # --- plotting ---
    fig, ax = plt.subplots(1, 1, sharex=True, sharey=True,
                        figsize=(double_textwidth, single_textwidth))

    xvals = np.log10(1 + cfg.zsample)

    for i in range(num_bins):
        idx_start = i * splitsize
        idx_end = (i + 1) * splitsize

        # S0
        average_chunk_S0 = np.average(MAHs_sorted[idx_start:idx_end], axis=0)
        color_S0 = cmap(norm(arr_meds[i]))
        ax.plot(xvals, average_chunk_S0, color=color_S0, lw=1.5)


    ax.set_yscale("log")  # crucial for absolute mass
    ax.set_xlim(0,0.8)
    ax.set_ylim(1e-2, 10)

    # --- colorbar ---
    plt.subplots_adjust(top=0.98)
    cbar_ax = fig.add_axes([0.17, 1.11, 0.7, 0.02])

    cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
    cbar.set_label(arr_label, loc="center")

    ax.set_xlabel("log (1 + z)")
    ax.set_ylabel("N$_{\\rm sub}$")
    # plt.tight_layout()
    plt.show()


In [ ]:
testr = average_Nacc("../../data/vdb/13.4_files.h5", regime="rvir")

In [ ]:
MAH_split(testr["Nusb_mat"], testr["log1pz50"], "log (1+z50)")

In [ ]:
MAH_split(testr["Nusb_mat"], testr["logc"], "log c")

In [ ]:
testw = average_Nacc("../../data/vdb/13.4_files.h5", regime="rvir", order="all")
testr = average_Nacc("../../data/vdb/13.4_files.h5", regime="rvir", order=1)
testa = average_Nacc("../../data/vdb/13.4_files.h5", regime="rvir", order=2)

In [ ]:
plt.plot(np.log10(1+cfg.zsample), testw["Nsub_ave"], color="C0", label='k=all')
plt.plot(np.log10(1+cfg.zsample), testr["Nsub_ave"], color="C1", label='k=1')
plt.plot(np.log10(1+cfg.zsample), testa["Nsub_ave"], color="C2", label='k=2')

plt.fill_between(np.log10(1+cfg.zsample), testw["Nsub_ave"] - testw["Nsub_std"], testw["Nsub_ave"] + testw["Nsub_std"], alpha=0.1, color="C0")
plt.fill_between(np.log10(1+cfg.zsample), testr["Nsub_ave"] - testr["Nsub_std"], testr["Nsub_ave"] + testr["Nsub_std"], alpha=0.1, color="C1")
plt.fill_between(np.log10(1+cfg.zsample), testa["Nsub_ave"] - testa["Nsub_std"], testa["Nsub_ave"] + testa["Nsub_std"], alpha=0.1, color="C2")

plt.yscale("log")
plt.xlim(0,1)
plt.ylim(1e-2, 1e2)
plt.legend(framealpha=1)

plt.xlabel("log (1+z)")
plt.ylabel("< N$_{\\rm sub}$ >")
plt.show()

In [ ]:
testw = average_facc("../../data/vdb/13.4_files.h5", regime="withering")
testr = average_facc("../../data/vdb/13.4_files.h5", regime="rvir")
testa = average_facc("../../data/vdb/13.4_files.h5", regime="artificial")

x = np.log10(1 + cfg.zsample)

plt.plot(x, testw["fsub_ave"], color="C0", label='survivors')
plt.plot(x, testr["fsub_ave"], color="C1", label='within R$_{\\rm vir}$')
plt.plot(x, testa["fsub_ave"], color="C2", label='resilient to\nartificial disruption')

plt.fill_between(x, testw["fsub_p16"], testw["fsub_p84"], alpha=0.1, color="C0")
plt.fill_between(x, testr["fsub_p16"], testr["fsub_p84"], alpha=0.1, color="C1")
plt.fill_between(x, testa["fsub_p16"], testa["fsub_p84"], alpha=0.1, color="C2")

plt.yscale("log")
plt.ylim(1e-3, 1)
plt.legend(framealpha=1)
plt.xlabel("log (1+z)")
plt.ylabel(r"$\langle f_{\rm sub} \rangle$")
plt.show()

In [ ]:
testw_shmf = average_SHMF("../../data/vdb/13.4_files.h5", regime="withering")
testr_shmf = average_SHMF("../../data/vdb/13.4_files.h5", regime="rvir")
testa_shmf = average_SHMF("../../data/vdb/13.4_files.h5", regime="artificial")

In [ ]:
fig, ax = plt.subplots()

for result, color, label in zip(
    [testw_shmf, testr_shmf, testa_shmf],
    ["C0", "C1", "C2"],
    ["survivors", "within R$_{\\rm vir}$", "with \nartificial disruption"]
):
    ax.plot(result["log_mM"], result["shmf_ave"], color=color, label=label)

xsmooth = np.linspace(-3, 0)
ax.plot(xsmooth, (10**xsmooth)**(-0.88), color='red', ls="--")
            
ax.set_yscale("log")
ax.set_xlabel(r"$\log(m/M_{\rm vir})$")
ax.set_ylabel(r"$dN/d\log(m/M_{\rm vir})$")
ax.legend(framealpha=1)
plt.show()